In [1]:
import os
import sys
import site
import re
import urllib.request
import zipfile
import base64
import shutil
import torch
from google.colab import drive
import numpy as np

# 0. CRITICAL GPU CHECK (Updated for CPU Fallback)
print("==================================================")
if torch.cuda.is_available():
    print(f"✅ GPU DETECTED: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: NO GPU DETECTED! Falling back to CPU.")
    print("⚠️ Training a Transformer on a CPU will be extremely slow.")
    print("⚠️ Applying Ultra-Fast CPU configuration for technical assessment completion.")
print("==================================================\n")

# 1. Mount Drive (Force remount to prevent Transport Endpoint errors)
drive.mount('/content/drive', force_remount=True)

# 2. Strictly enforce working directory to target the real repository
PROJECT_DIR = '/content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"Current Working Directory: {os.getcwd()}")

# 3. Dataset Verification & LOCAL SSD DOWNLOAD (The Ultimate Speed Fix)
# We bypass Google Drive entirely and download the zips directly from Hugging Face.
# We use Base64 encoding to completely hide the URL from Colab's buggy auto-markdown UI parser.
local_dataset_dir = '/content/dataset'

# Encoded: [https://huggingface.co/datasets/doron333/change-detection-dataset/resolve/main/](https://huggingface.co/datasets/doron333/change-detection-dataset/resolve/main/)
encoded_url = b'aHR0cHM6Ly9odWdnaW5nZmFjZS5jby9kYXRhc2V0cy9kb3JvbjMzMy9jaGFuZ2UtZGV0ZWN0aW9uLWRhdGFzZXQvcmVzb2x2ZS9tYWluLw=='
hf_base_url = base64.b64decode(encoded_url).decode('utf-8')

print(f"\n⏳ Downloading and Extracting Dataset directly from Hugging Face to Local SSD...")
os.makedirs(local_dataset_dir, exist_ok=True)

for split in ['train', 'val', 'test']:
    zip_path = os.path.join(local_dataset_dir, f"{split}.zip")
    extract_path = os.path.join(local_dataset_dir, split)

    # Check if this specific split is fully extracted and properly structured
    if not os.path.exists(os.path.join(extract_path, 'pre-event')):
        print(f"   -> Downloading {split}.zip (This takes ~15 seconds)...")
        try:
            urllib.request.urlretrieve(hf_base_url + split + ".zip", zip_path)

            print(f"   -> Extracting {split}.zip...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)

            # Clean up the zip file to save Colab disk space
            os.remove(zip_path)

            # Fix nested folder structure if it exists (e.g., train/train/pre-event)
            nested_dir = os.path.join(extract_path, split)
            if os.path.exists(nested_dir) and os.path.isdir(nested_dir):
                print(f"   -> Fixing nested folder structure for {split}...")
                for item in os.listdir(nested_dir):
                    source = os.path.join(nested_dir, item)
                    destination = os.path.join(extract_path, item)
                    # Move files/folders up one level
                    shutil.move(source, destination)
                # Remove the now-empty nested directory
                os.rmdir(nested_dir)

        except Exception as e:
            print(f"⚠️ Error downloading/extracting {split}.zip: {e}")
    else:
        print(f"   -> {split} already extracted.")

print("✅ Dataset successfully ready at Local Storage!")
dataset_dir = local_dataset_dir


# 4. FORCE APPLY PATCHES
site_packages = '/usr/local/lib/python3.12/dist-packages'

# A. Patch mmcv version checks
for lib in ['mmseg', 'mmdet', 'opencd']:
    path = "opencd/__init__.py" if lib == 'opencd' else os.path.join(site_packages, lib, "__init__.py")
    if os.path.exists(path):
        with open(path, 'r') as f: content = f.read()
        content = content.replace("'2.2.0'", "'2.3.0'").replace('"2.2.0"', '"2.3.0"')
        with open(path, 'w') as f: f.write(content)

# B. Apply the mmpretrain BLIP 'NoneType' bug patch
blip_path = os.path.join(site_packages, "mmpretrain", "models", "multimodal", "blip", "language_model.py")
if os.path.exists(blip_path):
    with open(blip_path, 'r') as f: content = f.read()
    content = content.replace('PreTrainedModel = None', 'PreTrainedModel = object')
    with open(blip_path, 'w') as f: f.write(content)

# C. Fix registry scope cross-talk, Optimizer Weight-Tying Bug, AND Validation Padding Bug
config_path = 'configs/changeformer/custom_disaster.py'
if os.path.exists(config_path):
    with open(config_path, 'r') as f: content = f.read()

    # Fix registry scope cross-talk
    content = content.replace("type='mmseg.PseudoSiamChangeFormer'", "type='PseudoSiamChangeFormer'")
    content = content.replace("type='PseudoSiamChangeFormer'", "type='mmseg.PseudoSiamChangeFormer'")

    # Fix PyTorch Optimizer Error AND the missing parenthesis SyntaxError
    safe_wrapper = "optim_wrapper = dict(\n    type='mmengine.OptimWrapper',\n    optimizer=dict(type='AdamW', lr=0.0001, weight_decay=0.01))\n\n"
    content = re.sub(r"optim_wrapper\s*=\s*dict\([\s\S]*?(?=# Learning Rate Scheduler)", safe_wrapper, content)

    # FIX FOR Validation Loop AssertionError: Tell the preprocessor that Segformer uses a stride of 32
    if "size_divisor=32" not in content:
        content = content.replace("seg_pad_val=255)", "seg_pad_val=255,\n        size_divisor=32)")

    with open(config_path, 'w') as f: f.write(content)

# D. Dynamically update the config file to point to the correct dataset paths & subfolders
dataset_config_path = 'configs/_base_/datasets/disaster_hetero_cd.py'
if os.path.exists(dataset_config_path):
    with open(dataset_config_path, 'r') as f: content = f.read()
    # Forcefully update base path using regex to catch any previous Google Drive paths!
    content = re.sub(r"data_root\s*=\s*['\"].*?['\"]", f"data_root = '{dataset_dir}'", content)
    # Update train subfolders
    content = content.replace("'train/pre'", "'train/pre-event'")
    content = content.replace("'train/post'", "'train/post-event'")
    content = content.replace("'train/label'", "'train/target'")
    # Update val subfolders
    content = content.replace("'val/pre'", "'val/pre-event'")
    content = content.replace("'val/post'", "'val/post-event'")
    content = content.replace("'val/label'", "'val/target'")
    # Update test subfolders
    content = content.replace("'test/pre'", "'test/pre-event'")
    content = content.replace("'test/post'", "'test/post-event'")
    content = content.replace("'test/label'", "'test/target'")

    # FIX FOR ValueError: Tell Open-CD to look for .tif files instead of default .png files
    if "img_suffix='.tif'" not in content:
        content = content.replace("data_root=data_root,", "data_root=data_root,\n        img_suffix='.tif',\n        seg_map_suffix='.tif',")

    # FIX FOR CUDA Assert Error: Inject our new BinarizeLabels transform into the training/test pipelines
    if "dict(type='BinarizeLabels')" not in content:
        content = content.replace("dict(type='MultiImgLoadAnnotations'),", "dict(type='MultiImgLoadAnnotations'),\n    dict(type='BinarizeLabels'),")

    # FIX FOR METRICS: Force Open-CD to calculate Precision, Recall, and F1-Score in addition to IoU
    if "iou_metrics=['mIoU']" in content:
        content = content.replace("iou_metrics=['mIoU']", "iou_metrics=['mIoU', 'mFscore']")

    with open(dataset_config_path, 'w') as f: f.write(content)

# E. FIX FOR Tensor 1024 ValueError & CUDA Index Error: Custom Dataloader and Label Binarizer
loader_code = """import numpy as np
import rasterio
from mmcv.transforms import BaseTransform
from opencd.registry import TRANSFORMS

@TRANSFORMS.register_module()
class LoadHeteroImagesFromFile(BaseTransform):
    \"\"\"Custom Loader for Heterogeneous EO-SAR Image Pairs.\"\"\"

    def transform(self, results: dict) -> dict:
        # Safely unpack Open-CD's list-based paths
        if isinstance(results['img_path'], list):
            pre_path = results['img_path'][0]
            post_path = results['img_path'][1]
        else:
            pre_path = results['img_path']
            post_path = results['img_path2']

        # 1. Load EO (Pre-event)
        with rasterio.open(pre_path) as src:
            eo = src.read().transpose(1, 2, 0).astype(np.float32)
            if eo.max() > 255:
                eo = eo / 10000.0  # Sentinel-2 scaling
            else:
                eo = eo / 255.0
            eo = np.clip(eo, 0, 1)

        # 2. Load SAR (Post-event)
        with rasterio.open(post_path) as src:
            sar = src.read().transpose(1, 2, 0).astype(np.float32)
            sar_db = 10 * np.log10(sar + 1e-8)
            sar_min, sar_max = np.min(sar_db), np.max(sar_db)
            if sar_max > sar_min:
                sar_norm = (sar_db - sar_min) / (sar_max - sar_min)
            else:
                sar_norm = sar_db

        # 3. Package as a LIST! Do NOT concatenate here.
        results['img'] = [eo, sar_norm]
        results['img_shape'] = eo.shape[:2]
        results['ori_shape'] = eo.shape[:2]

        return results

@TRANSFORMS.register_module()
class BinarizeLabels(BaseTransform):
    \"\"\"Forces ground truth masks to be strictly 0 and 1, preventing CUDA out-of-bounds errors.\"\"\"
    def transform(self, results: dict) -> dict:
        if 'gt_seg_map' in results:
            gt = results['gt_seg_map']
            # Convert visual masks (255) or multi-class damage indexes (2, 3, 4) strictly to Class 1
            gt = np.where(gt == 255, 1, gt)
            gt = np.where(gt > 1, 1, gt)
            results['gt_seg_map'] = gt
        return results
"""
os.makedirs('opencd/datasets/transforms', exist_ok=True)
with open('opencd/datasets/transforms/hetero_loading.py', 'w') as f:
    f.write(loader_code)

# F. GPU TRAINING OPTIMIZATION: Configure for 5,000 Iterations
if os.path.exists(config_path):
    with open(config_path, 'r') as f: content = f.read()

    # 1. Set Total Training Length
    # We catch the original 40k or the previous CPU fallback 100
    content = content.replace("max_iters=40000", "max_iters=5000")
    content = content.replace("max_iters=100", "max_iters=5000")

    # 2. Set Warmup Phase (10% of total run = 500 iterations)
    # This ensures the Transformer doesn't explode at the start
    content = content.replace("end=1500", "end=500")
    content = content.replace("end=20", "end=500")
    content = content.replace("begin=1500", "begin=500")
    content = content.replace("begin=20", "begin=500")

    # 3. Sync Learning Rate Scheduler (T_max = Total - Warmup = 4500)
    content = content.replace("T_max=38500", "T_max=4500")
    content = content.replace("T_max=80", "T_max=4500")
    content = content.replace("T_max=3500", "T_max=4500")

    # 4. Set Validation and Checkpoint Intervals
    # We validate every 500 steps to monitor mIoU progress
    content = content.replace("val_interval=4000", "val_interval=500")
    content = content.replace("val_interval=50", "val_interval=500")
    content = content.replace("interval=4000", "interval=500")
    content = content.replace("interval=50", "interval=500")

    with open(config_path, 'w') as f: f.write(content)
    print("✅ Configuration optimized for 5,000 GPU iterations.")

# G. FIX FOR COLAB RAM CRASHES & DATALOADER DEADLOCKS:
if os.path.exists(dataset_config_path):
    with open(dataset_config_path, 'r') as f: content = f.read()
    # Drop workers from 4 to 2 (Colab CPU limits)
    content = content.replace("num_workers=4", "num_workers=2")
    # Drop batch size from 16 to 8 (Colab RAM limits)
    content = content.replace("batch_size=16", "batch_size=8")

    # PREVENT C++ DEADLOCKS: Switch from DefaultSampler to InfiniteSampler so Rasterio threads never die
    content = content.replace("sampler=dict(type='DefaultSampler', shuffle=True)", "sampler=dict(type='InfiniteSampler', shuffle=True)")

    with open(dataset_config_path, 'w') as f: f.write(content)

# H. FIX FOR CUDA DEVICE-SIDE ASSERT & TYPE ERRORS:
if os.path.exists(config_path):
    with open(config_path, 'r') as f: content = f.read()
    safe_loss = "loss_decode=[\n            dict(type='mmseg.CrossEntropyLoss', use_sigmoid=False, loss_weight=1.0, loss_name='loss_ce'),\n            dict(type='mmseg.DiceLoss', loss_weight=1.0, loss_name='loss_dice')\n        ]),"
    content = re.sub(r"loss_decode=\[[\s\S]*?\]\),", safe_loss, content)
    with open(config_path, 'w') as f: f.write(content)

# I. CPU FALLBACK FIX: SyncBN crashes on CPU. Switch to standard BN.
if not torch.cuda.is_available():
    if os.path.exists(config_path):
        with open(config_path, 'r') as f: content = f.read()
        content = content.replace("type='SyncBN'", "type='BN'")
        with open(config_path, 'w') as f: f.write(content)

# 5. Check if Colab environment needs restoration
try:
    import mmengine
    import mmcv
    import transformers
    import rasterio
    print("\n✅ Environment dependencies and patches are intact. Ready to train.")
except ImportError:
    print("\n⚠️ New Colab session detected. Restoring OpenMMLab dependencies...")
    p_host = "download" + "." + "pytorch" + ".org"
    pt_url = f"https://{p_host}/whl/cu121"
    m_host = "download" + "." + "openmmlab" + ".com"
    mmcv_url = f"https://{m_host}/mmcv/dist/cu121/torch2.3.0/index.html"

    !pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url {pt_url}
    !pip install --upgrade pip setuptools wheel ninja
    !pip install -U mmengine "mmpretrain>=1.0.0rc7"
    !pip install mmcv==2.2.0 -f {mmcv_url}
    !pip install "mmsegmentation>=1.2.2" "mmdet>=3.0.0" ftfy regex "transformers<4.36.0"
    !pip install rasterio tifffile albumentations
    !pip install -v -e .

⚠️ WARNING: NO GPU DETECTED! Falling back to CPU.
⚠️ Training a Transformer on a CPU will be extremely slow.
⚠️ Applying Ultra-Fast CPU configuration for technical assessment completion.

Mounted at /content/drive
Current Working Directory: /content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd

⏳ Downloading and Extracting Dataset directly from Hugging Face to Local SSD...
   -> train already extracted.
   -> val already extracted.
   -> test already extracted.
✅ Dataset successfully ready at Local Storage!
✅ Configuration optimized for 5,000 GPU iterations.


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(



✅ Environment dependencies and patches are intact. Ready to train.


In [ ]:
import shutil
import os
import re

PROJECT_DIR = '/content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd'
os.chdir(PROJECT_DIR)

# 1. FORCE LOCAL SSD & FIX LEGACY SYMLINK BUGS
# Fix Phase 2 base config data root
base_cfg_path = 'configs/_base_/datasets/disaster_hetero_cd.py'
if os.path.exists(base_cfg_path):
    with open(base_cfg_path, 'r') as f: content = f.read()
    content = re.sub(r"data_root\s*=\s*['\"].*?['\"]", "data_root = '/content/dataset'", content)
    with open(base_cfg_path, 'w') as f: f.write(content)
    print("✅ Base configuration explicitly locked to Local SSD (/content/dataset).")

# Scrub legacy '/content/opencd_safe' paths from main config
main_cfg_path = 'configs/changeformer/custom_disaster.py'
if os.path.exists(main_cfg_path):
    with open(main_cfg_path, 'r') as f: content = f.read()
    content = content.replace("'/content/opencd_safe/configs/_base_/", "'../_base_/")
    with open(main_cfg_path, 'w') as f: f.write(content)
    print("✅ Scrubbed legacy symlink paths from main configuration.")

# 2. CACHE WIPE: We MUST delete the old work_dir to clear any old Google Drive paths.
work_dir = './work_dirs/disaster_changeformer'
if os.path.exists(work_dir):
    print(f"🧹 Wiping old cached training directory: {work_dir}")
    shutil.rmtree(work_dir)
else:
    print("✨ No old cache found. Ready for a fresh start!")

# 3. Launch training! PYTHONPATH="." ensures Python can find opencd natively
!PYTHONPATH="." python tools/train.py configs/changeformer/custom_disaster.py \
    --work-dir ./work_dirs/disaster_changeformer

✅ Base configuration explicitly locked to Local SSD (/content/dataset).
✅ Scrubbed legacy symlink paths from main configuration.
🧹 Wiping old cached training directory: ./work_dirs/disaster_changeformer
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
05/12 18:03:56 - mmengine - INFO - 
--------------------------

In [2]:
import os
import re

PROJECT_DIR = '/content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd'
os.chdir(PROJECT_DIR)

# --- FIX: Inject Custom HeteroVisHook with Native OpenCV Bypass ---

# 1. Update the Custom Hook
hook_file = 'opencd/datasets/transforms/hetero_loading.py'
if os.path.exists(hook_file):
    with open(hook_file, 'r') as f:
        hook_content = f.read()

    # Clean up any old HeteroVisHook to prevent file bloat
    if "class HeteroVisHook" in hook_content:
        hook_content = re.sub(r"@HOOKS\.register_module\(force=True\)\nclass HeteroVisHook[\s\S]*", "", hook_content)

    robust_hook = """\n
from mmseg.engine.hooks import SegVisualizationHook
from mmengine.registry import HOOKS
import os
import cv2
import numpy as np

@HOOKS.register_module(force=True)
class HeteroVisHook(SegVisualizationHook):
    def _after_iter(self, runner, batch_idx, data_batch, outputs, mode='val'):
        if getattr(self, 'draw', False) is False or mode == 'train':
            return

        interval = getattr(self, 'interval', 1)
        if self.every_n_inner_iters(batch_idx, interval):
            for data_sample in outputs:
                try:
                    img_path = data_sample.img_path
                    # Safely extract the EO (Pre-event) path
                    eo_path = img_path[0] if isinstance(img_path, (list, tuple)) else img_path

                    name = os.path.basename(eo_path) if eo_path else f'{mode}_{batch_idx}'

                    # Safely determine the output directory
                    out_dir = getattr(self, 'out_dir', None)
                    if out_dir is None and runner.work_dir is not None:
                        out_dir = os.path.join(runner.work_dir, 'predictions')

                    if out_dir is not None:
                        os.makedirs(out_dir, exist_ok=True)
                        out_file = os.path.join(out_dir, f'{name}_{runner.iter}.png')

                        img = cv2.imread(eo_path)
                        if img is None:
                            img = np.zeros((256, 256, 3), dtype=np.uint8)

                        panels = []

                        # THE ULTIMATE FIX: Bypass the brittle Visualizer API completely!
                        # We use native OpenCV to draw the masks and save them side-by-side.

                        # 1. Draw Ground Truth (Green Overlay)
                        if getattr(self, 'draw_gt', True) and hasattr(data_sample, 'gt_sem_seg'):
                            gt = data_sample.gt_sem_seg.data.squeeze().cpu().numpy()
                            gt_overlay = img.copy()
                            gt_overlay[gt == 1] = [0, 255, 0] # BGR Green
                            panels.append(cv2.addWeighted(gt_overlay, 0.6, img, 0.4, 0))

                        # 2. Draw Prediction (Red Overlay)
                        if getattr(self, 'draw_pred', True) and hasattr(data_sample, 'pred_sem_seg'):
                            pred = data_sample.pred_sem_seg.data.squeeze().cpu().numpy()
                            pred_overlay = img.copy()
                            pred_overlay[pred == 1] = [0, 0, 255] # BGR Red
                            panels.append(cv2.addWeighted(pred_overlay, 0.6, img, 0.4, 0))

                        if not panels:
                            panels.append(img)

                        # Concatenate side-by-side and save
                        vis_img = np.concatenate(panels, axis=1)
                        cv2.imwrite(out_file, vis_img)

                except Exception as e:
                    # Fail silently to guarantee the evaluation loop never crashes
                    pass
"""
    with open(hook_file, 'w') as f:
        f.write(hook_content + robust_hook)
    print("🔧 Patched Transform: Applied native OpenCV HeteroVisHook bypass.")

# 2. Inject the hook into the configuration
config_path = 'configs/changeformer/custom_disaster.py'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        content = f.read()

    # Clean up any existing/duplicate visualization hooks to prevent SyntaxErrors
    if "visualization=" in content:
        content = re.sub(r",\s*visualization=dict\([^)]*\)", "", content)

    # Safely inject the new HeteroVisHook
    if "HeteroVisHook" not in content:
        content = re.sub(
            r"(logger=dict\(type='LoggerHook'[^)]*\))",
            r"\1, visualization=dict(type='HeteroVisHook', draw=True, interval=1)",
            content
        )
        with open(config_path, 'w') as f:
            f.write(content)
        print("🔧 Patched Config: Activated HeteroVisHook for image generation.")
# ----------------------------------------------------

# Target the best checkpoint saved at the end of your training
best_checkpoint = './work_dirs/disaster_changeformer/best_mIoU_iter_5000.pth'

if os.path.exists(best_checkpoint):
    print(f"🎯 Found optimal trained weights: {best_checkpoint}")
    print("📊 Launching Final Inference on the UNSEEN Test Set...")
    print("⏳ This will take a few minutes to process all test images...\n")

    # Run test script and save output images to a 'predictions' folder
    !PYTHONPATH="." python tools/test.py \
        configs/changeformer/custom_disaster.py \
        {best_checkpoint} \
        --show-dir ./work_dirs/predictions

    print("\n✅ Inference Complete!")
    print("👉 Go to your Google Drive: 'open-cd/work_dirs/predictions' to see the visual results.")
else:
    print("⚠️ Checkpoint not found. Please verify the path.")

🔧 Patched Transform: Applied native OpenCV HeteroVisHook bypass.
🔧 Patched Config: Activated HeteroVisHook for image generation.
🎯 Found optimal trained weights: ./work_dirs/disaster_changeformer/best_mIoU_iter_5000.pth
📊 Launching Final Inference on the UNSEEN Test Set...
⏳ This will take a few minutes to process all test images...

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytre

In [ ]:
from google.colab import drive
import os

# 1. Re-mount Drive
drive.mount('/content/drive')

# 2. Go to project folder
PROJECT_DIR = '/content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd'
os.chdir(PROJECT_DIR)
print("✅ Drive re-mounted. You can now run Cell 4!")

In [3]:
import json
import os
import glob

PROJECT_DIR = '/content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd'
os.chdir(PROJECT_DIR)

print("🛰️ --- GalaxEye Change Detection Final Performance Report --- 🛰️\n")

# MMEngine often saves the test logs in the default config folder (custom_disaster) instead of the training folder.
# We use a recursive search to dynamically find the newest JSON log anywhere inside work_dirs!
json_logs = glob.glob('./work_dirs/**/*.json', recursive=True)

test_logs = []
for log in json_logs:
    try:
        with open(log, 'r') as f:
            content = f.read()
            # Verify the log actually contains the final test metrics
            if 'aAcc' in content and 'mIoU' in content:
                test_logs.append(log)
    except:
        pass

if test_logs:
    latest_log = max(test_logs, key=os.path.getmtime)
    print(f"📄 Successfully Parsed Log: {latest_log}\n")

    try:
        with open(latest_log, 'r') as f:
            for line in f:
                data = json.loads(line)
                # We are looking for the final test evaluation dictionary
                if 'aAcc' in data and 'mIoU' in data:
                    print("📌 OVERALL METRICS (Including Background):")
                    print(f"   Overall Accuracy (aAcc): {data.get('aAcc', 0):.2f}%")
                    print(f"   Mean F1-Score (mFscore): {data.get('mFscore', 0):.2f}%")
                    print(f"   Mean IoU (mIoU):         {data.get('mIoU', 0):.2f}%\n")
                    print("-" * 50)

                    # Look for class-specific metrics if they were logged
                    print("🚨 CRITICAL METRICS FOR CLASS 1 (DAMAGED/CHANGED):")
                    print("If these are 0.0, the model experienced 'Mode Collapse' due to the 5,000 iteration limit and severe class imbalance.")
                    print(f"   F1-Score:  {data.get('Ret-1-Fscore', 'See Console Output'):.2f}%" if isinstance(data.get('Ret-1-Fscore'), float) else f"   F1-Score:  See Console Output")
                    print(f"   Recall:    {data.get('Ret-1-Recall', 'See Console Output'):.2f}%" if isinstance(data.get('Ret-1-Recall'), float) else f"   Recall:    See Console Output")
                    print(f"   Precision: {data.get('Ret-1-Precision', 'See Console Output'):.2f}%" if isinstance(data.get('Ret-1-Precision'), float) else f"   Precision: See Console Output")
                    print("-" * 50)
                    print("\n👉 ACTION: Use these numbers to justify your architectural choices in your final report!")
                    break
    except Exception as e:
        print(f"Could not parse log file: {e}")
else:
    print("⚠️ No test logs containing metrics were found.")
    print("Please manually copy the 'changed' row from the printout in Cell 3.")

🛰️ --- GalaxEye Change Detection Final Performance Report --- 🛰️

📄 Successfully Parsed Log: ./work_dirs/custom_disaster/20260513_084608/20260513_084608.json

📌 OVERALL METRICS (Including Background):
   Overall Accuracy (aAcc): 83.93%
   Mean F1-Score (mFscore): 91.27%
   Mean IoU (mIoU):         41.97%

--------------------------------------------------
🚨 CRITICAL METRICS FOR CLASS 1 (DAMAGED/CHANGED):
If these are 0.0, the model experienced 'Mode Collapse' due to the 5,000 iteration limit and severe class imbalance.
   F1-Score:  See Console Output
   Recall:    See Console Output
   Precision: See Console Output
--------------------------------------------------

👉 ACTION: Use these numbers to justify your architectural choices in your final report!
